# AlertaVida | Pipeline de Big Data

## 1. Apresentação do projeto

O AlertaVida é uma solução CitizenTech voltada ao apoio na identificação e no encaminhamento de possíveis acidentes de trânsito por meio de câmeras urbanas.

Este notebook implementa o pipeline de Big Data responsável por integrar, processar, transformar e disponibilizar dados para análises e consumo pelo MVP.

### Objetivos
- Integrar dados históricos de acidentes da ARTESP.
- Processar eventos simulados gerados pelo AlertaVida.
- Aplicar regras de qualidade e tratamento de dados.
- Organizar os dados nas camadas Bronze, Silver e Gold.
- Disponibilizar indicadores e informações para o dashboard.

### Fontes de dados
- **ARTESP:** dados abertos históricos de acidentes em rodovias estaduais, referentes a 2025.
- **AlertaVida:** eventos simulados de possíveis ocorrências gerados pela DAG do Airflow.

### Tecnologias utilizadas
- Databricks Free Edition
- Apache Spark / PySpark
- SQL
- Apache Airflow
- Python

### Arquitetura

O pipeline utiliza a arquitetura Medallion:

- **Bronze:** armazenamento dos dados brutos.
- **Silver:** tratamento, padronização e validação.
- **Gold:** organização dos dados para análises e consumo pelo MVP.

> Os eventos gerados pelo AlertaVida são simulados e não representam ocorrências reais.

In [0]:
# 1. Criar o schema do projeto
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS workspace.alertavida
""")

# 2. Criar o Volume para receber os arquivos
spark.sql("""
    CREATE VOLUME IF NOT EXISTS workspace.alertavida.chegada
""")

# 3. Definir o caminho principal
base = "/Volumes/workspace/alertavida/chegada"

# 4. Criar pastas para organizar as fontes
pastas = [
    f"{base}/acidentes_artesp",
    f"{base}/eventos_alertavida",
    f"{base}/_controle"
]

for pasta in pastas:
    dbutils.fs.mkdirs(pasta)

print("Estrutura do AlertaVida criada!")

Estrutura do AlertaVida criada!


In [0]:
# Verificar estrutura criada

display(
    dbutils.fs.ls("/Volumes/workspace/alertavida/chegada")
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/alertavida/chegada/_controle/,_controle/,0,1794370867649
dbfs:/Volumes/workspace/alertavida/chegada/acidentes_artesp/,acidentes_artesp/,0,1794370867649
dbfs:/Volumes/workspace/alertavida/chegada/eventos_alertavida/,eventos_alertavida/,0,1794370867650


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/alertavida/chegada/acidentes_artesp"))

display(dbutils.fs.ls("/Volumes/workspace/alertavida/chegada/eventos_alertavida"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/alertavida/chegada/acidentes_artesp/artesp_acidentes_2026-11-09.csv,artesp_acidentes_2026-11-09.csv,10012987,1794207818000
dbfs:/Volumes/workspace/alertavida/chegada/acidentes_artesp/artesp_acidentes_2026-11-10.csv,artesp_acidentes_2026-11-10.csv,10012987,1794342535000


path,name,size,modificationTime
dbfs:/Volumes/workspace/alertavida/chegada/eventos_alertavida/eventos_alertavida_2026-11-09.csv,eventos_alertavida_2026-11-09.csv,331,1794207843000
dbfs:/Volumes/workspace/alertavida/chegada/eventos_alertavida/eventos_alertavida_2026-11-10.csv,eventos_alertavida_2026-11-10.csv,331,1794342564000


## 2. Camada Bronze | Dados brutos

A camada Bronze representa a primeira etapa de armazenamento do pipeline AlertaVida. Nela, os dados recebidos das fontes são carregados em seu formato original, preservando as informações para processamento posterior.

### Tabelas criadas

- `bronze_artesp`: armazena os registros históricos de acidentes disponibilizados pela ARTESP.
- `bronze_eventos`: armazena os eventos simulados gerados pela DAG do Airflow.

### Estratégia de ingestão

A ingestão é realizada por meio do comando SQL `COPY INTO`, utilizando arquivos CSV armazenados no Volume do Databricks.

As colunas são inicialmente armazenadas como STRING, permitindo que a conversão e validação dos tipos sejam realizadas na camada Silver.

O `COPY INTO` identifica arquivos já processados, evitando a duplicação de registros em reexecuções com os mesmos arquivos.

### Resultado das duas cargas

Foram realizadas duas coletas em dias diferentes, com arquivos armazenados no Volume do Databricks.

- ARTESP: 88.268 registros brutos.
- Eventos AlertaVida: 6 registros.
- Total: 88.274 registros.

A base da ARTESP contém dados históricos de 2025. Por isso, os dois arquivos de coleta possuem o mesmo conteúdo histórico. A duplicidade é preservada na camada Bronze e tratada posteriormente na Silver.

In [0]:

# Tabela Bronze da ARTESP
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.alertavida.bronze_artesp (
    data STRING,
    hora STRING,
    concessionaria STRING,
    rodovia STRING,
    km STRING,
    sentido STRING,
    latitude STRING,
    longitude STRING,
    classe STRING,
    subclasse STRING,
    causa_provavel STRING,
    vitima_ilesa STRING,
    vitima_leve STRING,
    vitima_moderada STRING,
    vitima_grave STRING,
    vitima_fatal STRING,
    vitimas_sem_info STRING,
    veiculos_envolvidos STRING,
    visibilidade STRING,
    condicao_meteriologica STRING,
    municipio STRING,
    regiao_administrativa STRING,
    regional_der STRING,
    jurisdicao STRING
)
USING DELTA
""")

# Criar tabela Bronze dos eventos AlertaVida
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.alertavida.bronze_eventos (
    evento_id STRING,
    camera_id STRING,
    tipo_evento STRING,
    confianca STRING,
    latitude STRING,
    longitude STRING,
    status STRING,
    simulado STRING
)
USING DELTA
""")

print("Tabelas Bronze criadas com sucesso!")


Tabelas Bronze criadas com sucesso!


In [0]:
spark.sql("""
SHOW TABLES IN workspace.alertavida
""").show(truncate=False)

+----------+--------------------+-----------+
|database  |tableName           |isTemporary|
+----------+--------------------+-----------+
|alertavida|bronze_artesp       |false      |
|alertavida|bronze_eventos      |false      |
|alertavida|gold_dim_camera     |false      |
|alertavida|gold_dim_data       |false      |
|alertavida|gold_dim_tipo_evento|false      |
|alertavida|gold_fato_eventos   |false      |
|alertavida|gold_mvp_alertas    |false      |
|alertavida|silver_artesp       |false      |
|alertavida|silver_eventos      |false      |
|alertavida|vw_artesp_tratada   |false      |
+----------+--------------------+-----------+



In [0]:
# Carregar acidentes da ARTESP
spark.sql("""
COPY INTO workspace.alertavida.bronze_artesp
FROM '/Volumes/workspace/alertavida/chegada/acidentes_artesp/'
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'header' = 'true',
    'encoding' = 'UTF-8'
)
""")

# Carregar eventos simulados do AlertaVida
spark.sql("""
COPY INTO workspace.alertavida.bronze_eventos
FROM '/Volumes/workspace/alertavida/chegada/eventos_alertavida/'
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'header' = 'true',
    'encoding' = 'UTF-8'
)
""")

print("Primeira carga Bronze concluída!")

Primeira carga Bronze concluída!


In [0]:
%sql
SELECT COUNT(*) AS total
FROM workspace.alertavida.bronze_artesp;

total
88268


In [0]:
%sql
SELECT COUNT(*) AS total
FROM workspace.alertavida.bronze_eventos;

total
6


In [0]:
print("ARTESP:")
spark.sql("""
SELECT COUNT(*) AS total
FROM workspace.alertavida.bronze_artesp
""").show()

print("EVENTOS:")
spark.sql("""
SELECT COUNT(*) AS total
FROM workspace.alertavida.bronze_eventos
""").show()

ARTESP:
+-----+
|total|
+-----+
|88268|
+-----+

EVENTOS:
+-----+
|total|
+-----+
|    6|
+-----+



In [0]:
spark.sql("""
SELECT
    DATA,
    HORA,
    RODOVIA,
    KM,
    LATITUDE,
    LONGITUDE,
    CLASSE,
    MUNICIPIO
FROM workspace.alertavida.bronze_artesp
LIMIT 10
""").show(truncate=False)

+----------+--------+----------+------+------------+------------+-----------------------+-----------+
|DATA      |HORA    |RODOVIA   |KM    |LATITUDE    |LONGITUDE   |CLASSE                 |MUNICIPIO  |
+----------+--------+----------+------+------------+------------+-----------------------+-----------+
|2025-01-01|04:11:25|SPI102/330|1.0   |-22.87259184|-47.13185728|ATROPELAMENTO DE ANIMAL|CAMPINAS   |
|2025-01-01|05:22:05|SP330     |63.9  |-23.15435809|-46.93528033|COLISÃO                |JUNDIAÍ    |
|2025-01-01|06:51:06|SP330     |114.6 |-22.79553822|-47.22437095|COLISÃO                |SUMARÉ     |
|2025-01-01|09:12:55|SPM330E   |102.0 |-22.868355  |-47.132428  |TOMBAMENTO             |CAMPINAS   |
|2025-01-01|12:30:23|SP330     |22.217|-23.451131  |-46.78673472|CHOQUE                 |SÃO PAULO  |
|2025-01-01|16:02:14|SP348     |16.3  |-23.49443104|-46.7322371 |COLISÃO                |SÃO PAULO  |
|2025-01-01|16:26:57|SP348     |106.0 |-22.89643789|-47.22418427|COLISÃO          

## 3. Camada Silver | Tratamento e qualidade

A camada Silver recebe os dados brutos da Bronze e aplica padronização, conversão de tipos e regras de qualidade.

O objetivo é garantir que somente registros válidos sejam disponibilizados para as análises posteriores.

### 3.1. Tratamento dos dados da ARTESP

- Conversão de datas e valores numéricos.
- Padronização dos campos.
- Geração de identificador único por hash SHA-256.
- Validação de data e hora.
- Validação das coordenadas geográficas.
- Verificação dos campos essenciais.

**Resultados da validação:**
- Registros após deduplicação: 44.058.
- Datas e horários inválidos: 0.
- Registros com coordenadas inválidas: 1.
- Dados essenciais inválidos: 0.
- Registros válidos na Silver: 44.057.

Um registro foi rejeitado por não possuir coordenadas geográficas válidas. A deduplicação foi realizada por identificador SHA-256, mantendo um registro por acidente.

In [0]:

spark.sql("""
CREATE OR REPLACE TEMP VIEW vw_artesp_tratada AS

WITH base AS (
    SELECT
        sha2(
            concat_ws('|',
                coalesce(DATA, ''),
                coalesce(HORA, ''),
                coalesce(RODOVIA, ''),
                coalesce(KM, ''),
                coalesce(LATITUDE, ''),
                coalesce(LONGITUDE, ''),
                coalesce(CLASSE, ''),
                coalesce(MUNICIPIO, '')
            ),
            256
        ) AS acidente_id,

        try_cast(DATA AS DATE) AS data_acidente,
        HORA AS hora_acidente,
        CONCESSIONARIA AS concessionaria,
        RODOVIA AS rodovia,
        try_cast(KM AS DOUBLE) AS km,
        SENTIDO AS sentido,
        try_cast(LATITUDE AS DOUBLE) AS latitude,
        try_cast(LONGITUDE AS DOUBLE) AS longitude,
        CLASSE AS classe,
        SUBCLASSE AS subclasse,
        CAUSA_PROVAVEL AS causa_provavel,

        try_cast(VITIMA_ILESA AS INT) AS vitima_ilesa,
        try_cast(VITIMA_LEVE AS INT) AS vitima_leve,
        try_cast(VITIMA_MODERADA AS INT) AS vitima_moderada,
        try_cast(VITIMA_GRAVE AS INT) AS vitima_grave,
        try_cast(VITIMA_FATAL AS INT) AS vitima_fatal,
        try_cast(VITIMAS_SEM_INFO AS INT) AS vitimas_sem_info,

        VEICULOS_ENVOLVIDOS AS veiculos_envolvidos,
        VISIBILIDADE AS visibilidade,
        CONDICAO_METERIOLOGICA AS condicao_meteorologica,
        MUNICIPIO AS municipio,
        REGIAO_ADMINISTRATIVA AS regiao_administrativa,
        REGIONAL_DER AS regional_der,
        JURISDICAO AS jurisdicao

    FROM workspace.alertavida.bronze_artesp
),

classificada AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY acidente_id
            ORDER BY regional_der ASC NULLS LAST
        ) AS rn
    FROM base
)

SELECT
    acidente_id,
    data_acidente,
    hora_acidente,
    concessionaria,
    rodovia,
    km,
    sentido,
    latitude,
    longitude,
    classe,
    subclasse,
    causa_provavel,
    vitima_ilesa,
    vitima_leve,
    vitima_moderada,
    vitima_grave,
    vitima_fatal,
    vitimas_sem_info,
    veiculos_envolvidos,
    visibilidade,
    condicao_meteorologica,
    municipio,
    regiao_administrativa,
    regional_der,
    jurisdicao
FROM classificada
WHERE rn = 1
""")

spark.sql("""
SELECT COUNT(*) AS total
FROM vw_artesp_tratada
""").show()


+-----+
|total|
+-----+
|44058|
+-----+



In [0]:
%sql
SELECT COUNT(*) AS total
FROM vw_artesp_tratada;

total
44058


In [0]:
spark.sql("""
SELECT
    COUNT(*) AS total_registros,

    SUM(
        CASE
            WHEN data_acidente IS NULL
              OR hora_acidente IS NULL
              OR NOT (hora_acidente RLIKE '^([01][0-9]|2[0-3]):[0-5][0-9]:[0-5][0-9]$')
            THEN 1 ELSE 0
        END
    ) AS rejeitados_data_hora,

    SUM(
        CASE
            WHEN latitude IS NULL
              OR longitude IS NULL
              OR latitude NOT BETWEEN -90 AND 90
              OR longitude NOT BETWEEN -180 AND 180
            THEN 1 ELSE 0
        END
    ) AS rejeitados_coordenadas,

    SUM(
        CASE
            WHEN rodovia IS NULL OR TRIM(rodovia) = ''
              OR classe IS NULL OR TRIM(classe) = ''
            THEN 1 ELSE 0
        END
    ) AS rejeitados_dados_essenciais

FROM vw_artesp_tratada
""").show()

+---------------+--------------------+----------------------+---------------------------+
|total_registros|rejeitados_data_hora|rejeitados_coordenadas|rejeitados_dados_essenciais|
+---------------+--------------------+----------------------+---------------------------+
|          44058|                   0|                     1|                          0|
+---------------+--------------------+----------------------+---------------------------+



In [0]:
spark.sql("""
SELECT
    data_acidente,
    rodovia,
    classe,
    municipio,
    latitude,
    longitude
FROM vw_artesp_tratada
WHERE latitude IS NULL
   OR longitude IS NULL
   OR latitude NOT BETWEEN -90 AND 90
   OR longitude NOT BETWEEN -180 AND 180
""").show(truncate=False)

+-------------+----------+-----------------------+-------------+--------+---------+
|data_acidente|rodovia   |classe                 |municipio    |latitude|longitude|
+-------------+----------+-----------------------+-------------+--------+---------+
|2025-04-28   |SPI065/099|ATROPELAMENTO DE ANIMAL|NÃO INFORMADO|NULL    |NULL     |
+-------------+----------+-----------------------+-------------+--------+---------+



In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.alertavida.silver_artesp (
    acidente_id STRING,
    data_acidente DATE,
    hora_acidente STRING,
    concessionaria STRING,
    rodovia STRING,
    km DOUBLE,
    sentido STRING,
    latitude DOUBLE,
    longitude DOUBLE,
    classe STRING,
    subclasse STRING,
    causa_provavel STRING,
    vitima_ilesa INT,
    vitima_leve INT,
    vitima_moderada INT,
    vitima_grave INT,
    vitima_fatal INT,
    vitimas_sem_info INT,
    veiculos_envolvidos STRING,
    visibilidade STRING,
    condicao_meteorologica STRING,
    municipio STRING,
    regiao_administrativa STRING,
    regional_der STRING,
    jurisdicao STRING
)
USING DELTA
""")

spark.sql("""
MERGE INTO workspace.alertavida.silver_artesp AS destino
USING (
    SELECT *
    FROM vw_artesp_tratada
    WHERE data_acidente IS NOT NULL
      AND hora_acidente RLIKE '^([01][0-9]|2[0-3]):[0-5][0-9]:[0-5][0-9]$'
      AND latitude BETWEEN -90 AND 90
      AND longitude BETWEEN -180 AND 180
      AND rodovia IS NOT NULL
      AND TRIM(rodovia) <> ''
      AND classe IS NOT NULL
      AND TRIM(classe) <> ''
) AS origem
ON destino.acidente_id = origem.acidente_id

WHEN NOT MATCHED THEN INSERT *
""")

print("Total na Silver:")
spark.sql("""
SELECT COUNT(*) AS total
FROM workspace.alertavida.silver_artesp
""").show()

Total na Silver:
+-----+
|total|
+-----+
|44057|
+-----+



In [0]:
%sql
DESCRIBE TABLE workspace.alertavida.silver_artesp;

col_name,data_type,comment
acidente_id,string,null
data_acidente,date,null
hora_acidente,string,null
concessionaria,string,null
rodovia,string,null
km,double,null
sentido,string,null
latitude,double,null
longitude,double,null
classe,string,null


In [0]:

spark.sql("""
CREATE OR REPLACE VIEW workspace.alertavida.vw_artesp_tratada AS

WITH base AS (
    SELECT
        sha2(
            concat_ws('|',
                coalesce(DATA, ''),
                coalesce(HORA, ''),
                coalesce(RODOVIA, ''),
                coalesce(KM, ''),
                coalesce(LATITUDE, ''),
                coalesce(LONGITUDE, ''),
                coalesce(CLASSE, ''),
                coalesce(MUNICIPIO, '')
            ), 256
        ) AS acidente_id,

        try_cast(DATA AS DATE) AS data_acidente,
        HORA AS hora_acidente,
        CONCESSIONARIA AS concessionaria,
        RODOVIA AS rodovia,
        try_cast(KM AS DOUBLE) AS km,
        SENTIDO AS sentido,
        try_cast(LATITUDE AS DOUBLE) AS latitude,
        try_cast(LONGITUDE AS DOUBLE) AS longitude,
        CLASSE AS classe,
        SUBCLASSE AS subclasse,
        CAUSA_PROVAVEL AS causa_provavel,
        try_cast(VITIMA_ILESA AS INT) AS vitima_ilesa,
        try_cast(VITIMA_LEVE AS INT) AS vitima_leve,
        try_cast(VITIMA_MODERADA AS INT) AS vitima_moderada,
        try_cast(VITIMA_GRAVE AS INT) AS vitima_grave,
        try_cast(VITIMA_FATAL AS INT) AS vitima_fatal,
        try_cast(VITIMAS_SEM_INFO AS INT) AS vitimas_sem_info,
        VEICULOS_ENVOLVIDOS AS veiculos_envolvidos,
        VISIBILIDADE AS visibilidade,
        CONDICAO_METERIOLOGICA AS condicao_meteorologica,
        MUNICIPIO AS municipio,
        REGIAO_ADMINISTRATIVA AS regiao_administrativa,
        REGIONAL_DER AS regional_der,
        JURISDICAO AS jurisdicao

    FROM workspace.alertavida.bronze_artesp
),

classificada AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY acidente_id
            ORDER BY regional_der ASC NULLS LAST
        ) AS rn
    FROM base
)

SELECT * EXCEPT (rn)
FROM classificada
WHERE rn = 1
""")

spark.sql("""
CREATE OR REPLACE TABLE workspace.alertavida.silver_artesp
USING DELTA
AS
SELECT *
FROM workspace.alertavida.vw_artesp_tratada
WHERE data_acidente IS NOT NULL
  AND hora_acidente RLIKE '^([01][0-9]|2[0-3]):[0-5][0-9]:[0-5][0-9]$'
  AND latitude BETWEEN -90 AND 90
  AND longitude BETWEEN -180 AND 180
  AND rodovia IS NOT NULL
""")

spark.sql("""
SELECT COUNT(*) AS total
FROM workspace.alertavida.silver_artesp
""").show()


+-----+
|total|
+-----+
|44057|
+-----+



In [0]:
%sql
SELECT COUNT(*) AS ids_duplicados
FROM (
    SELECT acidente_id
    FROM workspace.alertavida.silver_artesp
    GROUP BY acidente_id
    HAVING COUNT(*) > 1
);

ids_duplicados
0


In [0]:
%sql
SELECT COUNT(*) AS duplicados
FROM (
    SELECT acidente_id
    FROM workspace.alertavida.silver_artesp
    GROUP BY acidente_id
    HAVING COUNT(*) > 1
);

duplicados
0


In [0]:
%sql
SELECT
    SUM(qtd) AS registros_duplicados,
    COUNT(*) AS ids_duplicados
FROM (
    SELECT acidente_id, COUNT(*) AS qtd
    FROM workspace.alertavida.silver_artesp
    GROUP BY acidente_id
    HAVING COUNT(*) > 1
);

registros_duplicados,ids_duplicados
null,0


In [0]:
%sql
SELECT *
FROM workspace.alertavida.silver_artesp
WHERE acidente_id IN (
    SELECT acidente_id
    FROM workspace.alertavida.silver_artesp
    GROUP BY acidente_id
    HAVING COUNT(*) > 1
)
LIMIT 10;

acidente_id,data_acidente,hora_acidente,concessionaria,rodovia,km,sentido,latitude,longitude,classe,subclasse,causa_provavel,vitima_ilesa,vitima_leve,vitima_moderada,vitima_grave,vitima_fatal,vitimas_sem_info,veiculos_envolvidos,visibilidade,condicao_meteorologica,municipio,regiao_administrativa,regional_der,jurisdicao


In [0]:
%sql
SELECT
    COUNT(*) AS total_linhas,
    COUNT(DISTINCT acidente_id) AS ids_unicos
FROM vw_artesp_tratada;

total_linhas,ids_unicos
44058,44058


In [0]:
spark.sql("""
SELECT *
FROM workspace.alertavida.bronze_eventos
""").show(truncate=False)

+-----------------+-----------+----------------------+---------+--------+---------+--------+--------+
|evento_id        |camera_id  |tipo_evento           |confianca|latitude|longitude|status  |simulado|
+-----------------+-----------+----------------------+---------+--------+---------+--------+--------+
|AV-2026-11-10-001|CAM-SIM-001|COLISAO               |0.94     |-23.3621|-46.7405 |PENDENTE|True    |
|AV-2026-11-10-002|CAM-SIM-002|POSSIVEL_ATROPELAMENTO|0.87     |-23.365 |-46.735  |PENDENTE|True    |
|AV-2026-11-10-003|CAM-SIM-003|QUEDA_DE_MOTOCICLISTA |0.91     |-23.359 |-46.738  |PENDENTE|True    |
|AV-2026-11-09-001|CAM-SIM-001|COLISAO               |0.94     |-23.3621|-46.7405 |PENDENTE|True    |
|AV-2026-11-09-002|CAM-SIM-002|POSSIVEL_ATROPELAMENTO|0.87     |-23.365 |-46.735  |PENDENTE|True    |
|AV-2026-11-09-003|CAM-SIM-003|QUEDA_DE_MOTOCICLISTA |0.91     |-23.359 |-46.738  |PENDENTE|True    |
+-----------------+-----------+----------------------+---------+--------+---------

### 3.2. Tratamento dos eventos do AlertaVida

Os eventos gerados pelo AlertaVida também passam por conversão de tipos e validações antes de serem disponibilizados na camada Silver.

**Tratamentos realizados:**
- Conversão da confiança para DOUBLE.
- Conversão de latitude e longitude para DOUBLE.
- Conversão do campo simulado para BOOLEAN.
- Validação dos campos obrigatórios: evento_id, camera_id e tipo_evento.
- Verificação da confiança, que deve estar entre 0 e 1.
- Validação dos limites geográficos das coordenadas.

**Resultados das regras de qualidade:**
- Campos obrigatórios inválidos: 0 registros.
- Confiança inválida: 0 registros.
- Coordenadas inválidas: 0 registros.

Após as validações, os dados são inseridos na tabela `silver_eventos` utilizando MERGE pela chave `evento_id`, evitando duplicações em novas execuções.

**Resultado:** 6 eventos válidos mantidos na camada Silver.

In [0]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW vw_eventos_tratados AS
SELECT
    evento_id,
    camera_id,
    tipo_evento,
    TRY_CAST(confianca AS DOUBLE) AS confianca,
    TRY_CAST(latitude AS DOUBLE) AS latitude,
    TRY_CAST(longitude AS DOUBLE) AS longitude,
    status,
    LOWER(simulado) = 'true' AS simulado
FROM workspace.alertavida.bronze_eventos
""")

spark.sql("""
SELECT
    COUNT(*) AS total_registros,

    SUM(
        CASE
            WHEN evento_id IS NULL OR TRIM(evento_id) = ''
              OR camera_id IS NULL OR TRIM(camera_id) = ''
              OR tipo_evento IS NULL OR TRIM(tipo_evento) = ''
            THEN 1 ELSE 0
        END
    ) AS rejeitados_campos_obrigatorios,

    SUM(
        CASE
            WHEN confianca IS NULL OR confianca NOT BETWEEN 0 AND 1
            THEN 1 ELSE 0
        END
    ) AS rejeitados_confianca,

    SUM(
        CASE
            WHEN latitude IS NULL OR longitude IS NULL
              OR latitude NOT BETWEEN -90 AND 90
              OR longitude NOT BETWEEN -180 AND 180
            THEN 1 ELSE 0
        END
    ) AS rejeitados_coordenadas

FROM vw_eventos_tratados
""").show()

+---------------+------------------------------+--------------------+----------------------+
|total_registros|rejeitados_campos_obrigatorios|rejeitados_confianca|rejeitados_coordenadas|
+---------------+------------------------------+--------------------+----------------------+
|              6|                             0|                   0|                     0|
+---------------+------------------------------+--------------------+----------------------+



In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.alertavida.silver_eventos (
    evento_id STRING,
    camera_id STRING,
    tipo_evento STRING,
    confianca DOUBLE,
    latitude DOUBLE,
    longitude DOUBLE,
    status STRING,
    simulado BOOLEAN
)
USING DELTA
""")

spark.sql("""
MERGE INTO workspace.alertavida.silver_eventos AS destino
USING (
    SELECT *
    FROM vw_eventos_tratados
    WHERE evento_id IS NOT NULL
      AND TRIM(evento_id) <> ''
      AND camera_id IS NOT NULL
      AND TRIM(camera_id) <> ''
      AND tipo_evento IS NOT NULL
      AND TRIM(tipo_evento) <> ''
      AND confianca BETWEEN 0 AND 1
      AND latitude BETWEEN -90 AND 90
      AND longitude BETWEEN -180 AND 180
) AS origem
ON destino.evento_id = origem.evento_id

WHEN NOT MATCHED THEN INSERT *
""")

print("Total na Silver de eventos:")
spark.sql("""
SELECT COUNT(*) AS total
FROM workspace.alertavida.silver_eventos
""").show()

Total na Silver de eventos:
+-----+
|total|
+-----+
|    6|
+-----+



## 4. Camada Gold | Modelagem analítica

Na camada Gold, os dados tratados na Silver são organizados em um modelo dimensional, facilitando consultas analíticas e a integração com o dashboard do AlertaVida.

### 4.1. Dimensões

Foram criadas três dimensões:

- **gold_dim_camera:** identifica as câmeras de origem dos eventos.
- **gold_dim_tipo_evento:** apresenta os tipos de ocorrência e suas descrições.
- **gold_dim_data:** organiza as informações temporais, como dia, mês e ano.

As dimensões permitem analisar os eventos por câmera, tipo de ocorrência e período.

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.alertavida.gold_dim_camera
USING DELTA
AS
SELECT DISTINCT
    camera_id,
    CASE
        WHEN camera_id LIKE 'CAM-SIM-%'
        THEN 'SIMULADA'
        ELSE 'REAL'
    END AS origem_camera
FROM workspace.alertavida.silver_eventos
""")

spark.sql("""
SELECT * FROM workspace.alertavida.gold_dim_camera
""").show(truncate=False)

+-----------+-------------+
|camera_id  |origem_camera|
+-----------+-------------+
|CAM-SIM-002|SIMULADA     |
|CAM-SIM-001|SIMULADA     |
|CAM-SIM-003|SIMULADA     |
+-----------+-------------+



In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.alertavida.gold_dim_tipo_evento
USING DELTA
AS
SELECT DISTINCT
    tipo_evento,
    CASE
        WHEN tipo_evento = 'COLISAO'
            THEN 'Colisão'
        WHEN tipo_evento = 'POSSIVEL_ATROPELAMENTO'
            THEN 'Possível atropelamento'
        WHEN tipo_evento = 'QUEDA_DE_MOTOCICLISTA'
            THEN 'Queda de motociclista'
        ELSE 'Outros'
    END AS descricao
FROM workspace.alertavida.silver_eventos
""")

spark.sql("""
SELECT *
FROM workspace.alertavida.gold_dim_tipo_evento
ORDER BY tipo_evento
""").show(truncate=False)

+----------------------+----------------------+
|tipo_evento           |descricao             |
+----------------------+----------------------+
|COLISAO               |Colisão               |
|POSSIVEL_ATROPELAMENTO|Possível atropelamento|
|QUEDA_DE_MOTOCICLISTA |Queda de motociclista |
+----------------------+----------------------+



In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.alertavida.gold_dim_data
USING DELTA
AS
SELECT DISTINCT
    TO_DATE(SUBSTRING(evento_id, 4, 10)) AS data,
    YEAR(TO_DATE(SUBSTRING(evento_id, 4, 10))) AS ano,
    MONTH(TO_DATE(SUBSTRING(evento_id, 4, 10))) AS mes,
    DAY(TO_DATE(SUBSTRING(evento_id, 4, 10))) AS dia,
    DATE_FORMAT(
        TO_DATE(SUBSTRING(evento_id, 4, 10)),
        'EEEE'
    ) AS dia_semana
FROM workspace.alertavida.silver_eventos
""")

spark.sql("""
SELECT *
FROM workspace.alertavida.gold_dim_data
""").show(truncate=False)

+----------+----+---+---+----------+
|data      |ano |mes|dia|dia_semana|
+----------+----+---+---+----------+
|2026-11-10|2026|11 |10 |Tuesday   |
|2026-11-09|2026|11 |9  |Monday    |
+----------+----+---+---+----------+



### 4.2. Tabela fato

A tabela `gold_fato_eventos` centraliza os eventos do AlertaVida e relaciona as informações das dimensões.

Ela armazena:
- Identificador do evento.
- Câmera de origem.
- Tipo de ocorrência.
- Data do evento.
- Nível de confiança da detecção.
- Latitude e longitude.
- Status da ocorrência.
- Indicador de evento simulado.

**Resultado:** 6 eventos registrados na tabela fato, relacionados às dimensões de câmera, tipo de evento e data.

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.alertavida.gold_fato_eventos
USING DELTA
AS
SELECT
    e.evento_id,
    c.camera_id,
    t.tipo_evento,
    d.data AS data_evento,
    e.confianca,
    e.latitude,
    e.longitude,
    e.status,
    e.simulado
FROM workspace.alertavida.silver_eventos e

INNER JOIN workspace.alertavida.gold_dim_camera c
    ON e.camera_id = c.camera_id

INNER JOIN workspace.alertavida.gold_dim_tipo_evento t
    ON e.tipo_evento = t.tipo_evento

INNER JOIN workspace.alertavida.gold_dim_data d
    ON TO_DATE(SUBSTRING(e.evento_id, 4, 10)) = d.data
""")

spark.sql("""
SELECT *
FROM workspace.alertavida.gold_fato_eventos
ORDER BY evento_id
""").show(truncate=False)

+-----------------+-----------+----------------------+-----------+---------+--------+---------+--------+--------+
|evento_id        |camera_id  |tipo_evento           |data_evento|confianca|latitude|longitude|status  |simulado|
+-----------------+-----------+----------------------+-----------+---------+--------+---------+--------+--------+
|AV-2026-11-09-001|CAM-SIM-001|COLISAO               |2026-11-09 |0.94     |-23.3621|-46.7405 |PENDENTE|true    |
|AV-2026-11-09-002|CAM-SIM-002|POSSIVEL_ATROPELAMENTO|2026-11-09 |0.87     |-23.365 |-46.735  |PENDENTE|true    |
|AV-2026-11-09-003|CAM-SIM-003|QUEDA_DE_MOTOCICLISTA |2026-11-09 |0.91     |-23.359 |-46.738  |PENDENTE|true    |
|AV-2026-11-10-001|CAM-SIM-001|COLISAO               |2026-11-10 |0.94     |-23.3621|-46.7405 |PENDENTE|true    |
|AV-2026-11-10-002|CAM-SIM-002|POSSIVEL_ATROPELAMENTO|2026-11-10 |0.87     |-23.365 |-46.735  |PENDENTE|true    |
|AV-2026-11-10-003|CAM-SIM-003|QUEDA_DE_MOTOCICLISTA |2026-11-10 |0.91     |-23.359 |-46

In [0]:
print("CONTAGEM DAS TABELAS GOLD")

for tabela in [
    "gold_dim_camera",
    "gold_dim_tipo_evento",
    "gold_dim_data",
    "gold_fato_eventos"
]:
    total = spark.table(
        f"workspace.alertavida.{tabela}"
    ).count()

    print(f"{tabela}: {total}")

print("\nVALIDAÇÃO DOS RELACIONAMENTOS")

spark.sql("""
SELECT
    COUNT(*) AS total_eventos,
    COUNT(DISTINCT evento_id) AS eventos_unicos,
    SUM(CASE WHEN simulado = true THEN 1 ELSE 0 END) AS eventos_simulados
FROM workspace.alertavida.gold_fato_eventos
""").show()

CONTAGEM DAS TABELAS GOLD
gold_dim_camera: 3
gold_dim_tipo_evento: 3
gold_dim_data: 2
gold_fato_eventos: 6

VALIDAÇÃO DOS RELACIONAMENTOS
+-------------+--------------+-----------------+
|total_eventos|eventos_unicos|eventos_simulados|
+-------------+--------------+-----------------+
|            6|             6|                6|
+-------------+--------------+-----------------+



In [0]:
%sql
SELECT 'gold_dim_camera' AS tabela, COUNT(*) AS total
FROM workspace.alertavida.gold_dim_camera

UNION ALL

SELECT 'gold_dim_tipo_evento', COUNT(*)
FROM workspace.alertavida.gold_dim_tipo_evento

UNION ALL

SELECT 'gold_dim_data', COUNT(*)
FROM workspace.alertavida.gold_dim_data

UNION ALL

SELECT 'gold_fato_eventos', COUNT(*)
FROM workspace.alertavida.gold_fato_eventos;

tabela,total
gold_dim_camera,3
gold_dim_tipo_evento,3
gold_dim_data,2
gold_fato_eventos,6


## 5. Consultas analíticas e resultados

Foram desenvolvidas cinco consultas SQL para demonstrar como os dados da camada Gold podem apoiar análises operacionais e estratégicas.

### 5.1. Visão geral dos alertas
Apresenta a quantidade total de alertas e sua distribuição por status, permitindo acompanhar as ocorrências registradas.

### 5.2. Alertas por tipo de ocorrência
Agrupa os eventos por tipo e calcula a confiança média das detecções, ajudando a identificar as ocorrências mais frequentes.

### 5.3. Alertas por câmera
Apresenta a quantidade de eventos e a confiança média por câmera, permitindo analisar a origem das detecções.

### 5.4. Acidentes históricos por município
Utiliza os dados da ARTESP para identificar os municípios com maior quantidade de acidentes registrados em 2025, incluindo informações sobre acidentes fatais e graves.

### 5.5. Análise de impacto regional
Relaciona os dados históricos de Caieiras, Cajamar e Franco da Rocha com os alertas simulados do AlertaVida, demonstrando como diferentes fontes podem ser utilizadas em uma análise regional.

**Observação:** os alertas do AlertaVida são simulados e não representam acidentes reais nem comprovam redução de ocorrências.

In [0]:
spark.sql("""
SELECT
    COUNT(*) AS total_alertas,

    SUM(
        CASE WHEN status = 'PENDENTE'
        THEN 1 ELSE 0 END
    ) AS pendentes,

    SUM(
        CASE WHEN status = 'CONFIRMADO'
        THEN 1 ELSE 0 END
    ) AS confirmados,

    SUM(
        CASE WHEN status = 'DESCARTADO'
        THEN 1 ELSE 0 END
    ) AS descartados

FROM workspace.alertavida.gold_fato_eventos
""").show()

+-------------+---------+-----------+-----------+
|total_alertas|pendentes|confirmados|descartados|
+-------------+---------+-----------+-----------+
|            6|        6|          0|          0|
+-------------+---------+-----------+-----------+



In [0]:
spark.sql("""
SELECT
    t.descricao AS tipo_ocorrencia,
    COUNT(*) AS quantidade,
    ROUND(AVG(f.confianca) * 100, 2) AS confianca_media_percentual

FROM workspace.alertavida.gold_fato_eventos f

INNER JOIN workspace.alertavida.gold_dim_tipo_evento t
    ON f.tipo_evento = t.tipo_evento

GROUP BY t.descricao
ORDER BY quantidade DESC
""").show(truncate=False)

+----------------------+----------+--------------------------+
|tipo_ocorrencia       |quantidade|confianca_media_percentual|
+----------------------+----------+--------------------------+
|Possível atropelamento|2         |87.0                      |
|Queda de motociclista |2         |91.0                      |
|Colisão               |2         |94.0                      |
+----------------------+----------+--------------------------+



In [0]:
spark.sql("""
SELECT
    c.camera_id,
    c.origem_camera,
    COUNT(f.evento_id) AS total_alertas,
    ROUND(AVG(f.confianca) * 100, 2) AS confianca_media_percentual

FROM workspace.alertavida.gold_dim_camera c

LEFT JOIN workspace.alertavida.gold_fato_eventos f
    ON c.camera_id = f.camera_id

GROUP BY
    c.camera_id,
    c.origem_camera

ORDER BY total_alertas DESC
""").show(truncate=False)

+-----------+-------------+-------------+--------------------------+
|camera_id  |origem_camera|total_alertas|confianca_media_percentual|
+-----------+-------------+-------------+--------------------------+
|CAM-SIM-002|SIMULADA     |2            |87.0                      |
|CAM-SIM-001|SIMULADA     |2            |94.0                      |
|CAM-SIM-003|SIMULADA     |2            |91.0                      |
+-----------+-------------+-------------+--------------------------+



In [0]:
spark.sql("""
SELECT
    municipio,
    COUNT(*) AS total_acidentes,

    SUM(vitima_fatal) AS total_vitimas_fatais,

    SUM(vitima_grave) AS total_vitimas_graves

FROM workspace.alertavida.silver_artesp

WHERE municipio IS NOT NULL
  AND TRIM(municipio) <> ''
  AND municipio <> 'NÃO INFORMADO'

GROUP BY municipio

ORDER BY total_acidentes DESC

LIMIT 10
""").show(truncate=False)

+---------------------+---------------+--------------------+--------------------+
|municipio            |total_acidentes|total_vitimas_fatais|total_vitimas_graves|
+---------------------+---------------+--------------------+--------------------+
|CAMPINAS             |2211           |58                  |136                 |
|SÃO PAULO            |1988           |45                  |48                  |
|SÃO BERNARDO DO CAMPO|1714           |48                  |30                  |
|COTIA                |1457           |26                  |37                  |
|JUNDIAÍ              |1333           |27                  |53                  |
|CUBATÃO              |1178           |26                  |18                  |
|RIBEIRÃO PRETO       |1153           |13                  |30                  |
|BARUERI              |1117           |17                  |26                  |
|ITU                  |850            |25                  |32                  |
|SOROCABA       

In [0]:
spark.sql("""
SELECT
    municipio,
    COUNT(*) AS total_acidentes,
    SUM(vitima_fatal) AS vitimas_fatais,
    SUM(vitima_grave) AS vitimas_graves
FROM workspace.alertavida.silver_artesp
WHERE municipio IN (
    'CAIEIRAS',
    'FRANCISCO MORATO',
    'FRANCO DA ROCHA',
    'JORDANÉSIA',
    'CAJAMAR'
)
GROUP BY municipio
ORDER BY total_acidentes DESC
""").show(truncate=False)

+---------------+---------------+--------------+--------------+
|municipio      |total_acidentes|vitimas_fatais|vitimas_graves|
+---------------+---------------+--------------+--------------+
|CAJAMAR        |297            |3             |8             |
|CAIEIRAS       |174            |9             |11            |
|FRANCO DA ROCHA|53             |0             |2             |
+---------------+---------------+--------------+--------------+



In [0]:
spark.sql("""
WITH historico AS (
    SELECT
        COUNT(*) AS acidentes_historicos,
        SUM(COALESCE(vitima_fatal, 0)) AS vitimas_fatais,
        SUM(COALESCE(vitima_grave, 0)) AS vitimas_graves
    FROM workspace.alertavida.silver_artesp
    WHERE municipio IN (
        'CAIEIRAS',
        'CAJAMAR',
        'FRANCO DA ROCHA'
    )
),

alertas AS (
    SELECT
        COUNT(*) AS total_alertas,
        SUM(
            CASE WHEN simulado = true
            THEN 1 ELSE 0 END
        ) AS alertas_simulados,
        ROUND(AVG(confianca) * 100, 2) AS confianca_media
    FROM workspace.alertavida.gold_fato_eventos
)

SELECT
    h.acidentes_historicos,
    h.vitimas_fatais,
    h.vitimas_graves,
    a.total_alertas,
    a.alertas_simulados,
    a.confianca_media
FROM historico h
CROSS JOIN alertas a
""").show()

+--------------------+--------------+--------------+-------------+-----------------+---------------+
|acidentes_historicos|vitimas_fatais|vitimas_graves|total_alertas|alertas_simulados|confianca_media|
+--------------------+--------------+--------------+-------------+-----------------+---------------+
|                 524|            12|            21|            6|                6|          90.67|
+--------------------+--------------+--------------+-------------+-----------------+---------------+



## 6. Integração com o MVP e proteção de dados

Para disponibilizar os dados ao dashboard do AlertaVida, foi criada a view `gold_mvp_alertas`, que reúne informações da tabela fato e das dimensões da camada Gold.

### 6.1. Dados disponibilizados

A view fornece:
- Identificador do evento.
- Referência pseudonimizada da câmera.
- Tipo de ocorrência.
- Data do evento.
- Confiança da detecção.
- Latitude e longitude.
- Status da ocorrência.
- Indicador de evento simulado.

### 6.2. Proteção de dados

O identificador da câmera é transformado em um hash SHA-256 antes de ser disponibilizado ao MVP, criando uma referência pseudonimizada e reduzindo a exposição de identificadores operacionais.

O projeto utiliza dados históricos públicos da ARTESP e eventos simulados, sem armazenar dados pessoais reais, como nomes, CPFs ou telefones.

### 6.3. Consumo pelo dashboard

O MVP utiliza uma API em Python com FastAPI para consultar os dados disponibilizados no Databricks.

A interface foi desenvolvida com React e permite visualizar os alertas registrados, seus detalhes e suas localizações geográficas em um mapa.

**Resultado:** o dashboard consegue consumir os dados da camada Gold e apresentar os seis eventos simulados do AlertaVida.

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW workspace.alertavida.gold_mvp_alertas AS
SELECT
    f.evento_id,
    SHA2(f.camera_id, 256) AS camera_ref_hash,
    t.descricao AS tipo_ocorrencia,
    f.data_evento,
    f.confianca,
    f.latitude,
    f.longitude,
    f.status,
    f.simulado
FROM workspace.alertavida.gold_fato_eventos f

INNER JOIN workspace.alertavida.gold_dim_tipo_evento t
    ON f.tipo_evento = t.tipo_evento
""")

spark.sql("""
SELECT *
FROM workspace.alertavida.gold_mvp_alertas
ORDER BY data_evento DESC
""").show(truncate=False)

+-----------------+----------------------------------------------------------------+----------------------+-----------+---------+--------+---------+--------+--------+
|evento_id        |camera_ref_hash                                                 |tipo_ocorrencia       |data_evento|confianca|latitude|longitude|status  |simulado|
+-----------------+----------------------------------------------------------------+----------------------+-----------+---------+--------+---------+--------+--------+
|AV-2026-11-10-002|06b14c14005ad4ce8b024addf6d1ea4ada9bc21dae3909d1fa80b907f46e88c0|Possível atropelamento|2026-11-10 |0.87     |-23.365 |-46.735  |PENDENTE|true    |
|AV-2026-11-10-001|faedca6a19701c1a851747cbf19af8246d5407a1f5f13a7bb8bee3e2d988fe04|Colisão               |2026-11-10 |0.94     |-23.3621|-46.7405 |PENDENTE|true    |
|AV-2026-11-10-003|554ab10d1ef505f90c2dd949677f48f77047f661a016a0d100d09b4b129c0a5a|Queda de motociclista |2026-11-10 |0.91     |-23.359 |-46.738  |PENDENTE|true    

In [0]:
%sql
SELECT *
FROM workspace.alertavida.gold_mvp_alertas
ORDER BY data_evento, evento_id;

evento_id,camera_ref_hash,tipo_ocorrencia,data_evento,confianca,latitude,longitude,status,simulado
AV-2026-11-09-001,faedca6a19701c1a851747cbf19af8246d5407a1f5f13a7bb8bee3e2d988fe04,Colisão,2026-11-09,0.94,-23.3621,-46.7405,PENDENTE,true
AV-2026-11-09-002,06b14c14005ad4ce8b024addf6d1ea4ada9bc21dae3909d1fa80b907f46e88c0,Possível atropelamento,2026-11-09,0.87,-23.365,-46.735,PENDENTE,true
AV-2026-11-09-003,554ab10d1ef505f90c2dd949677f48f77047f661a016a0d100d09b4b129c0a5a,Queda de motociclista,2026-11-09,0.91,-23.359,-46.738,PENDENTE,true
AV-2026-11-10-001,faedca6a19701c1a851747cbf19af8246d5407a1f5f13a7bb8bee3e2d988fe04,Colisão,2026-11-10,0.94,-23.3621,-46.7405,PENDENTE,true
AV-2026-11-10-002,06b14c14005ad4ce8b024addf6d1ea4ada9bc21dae3909d1fa80b907f46e88c0,Possível atropelamento,2026-11-10,0.87,-23.365,-46.735,PENDENTE,true
AV-2026-11-10-003,554ab10d1ef505f90c2dd949677f48f77047f661a016a0d100d09b4b129c0a5a,Queda de motociclista,2026-11-10,0.91,-23.359,-46.738,PENDENTE,true


## 7. Conclusão

O projeto AlertaVida demonstrou a construção de um pipeline de Big Data utilizando a arquitetura Medallion, organizando os dados nas camadas Bronze, Silver e Gold.

A integração entre Airflow e Databricks permitiu automatizar a coleta, validar e tratar os dados e disponibilizá-los para análises e consumo pelo MVP.

A utilização de dados públicos da ARTESP, combinados com eventos simulados, possibilitou demonstrar como diferentes fontes podem ser integradas para apoiar o monitoramento e a análise de possíveis ocorrências de trânsito.

O dashboard conectado à camada Gold representa a etapa de consumo dos dados, apresentando os alertas em uma interface visual com informações e localização geográfica.

**Resultado final:** um fluxo integrado de coleta, processamento, armazenamento, análise e visualização de dados, com possibilidade de evolução para fontes de eventos reais e autorizadas.